# 🐟 FishSizeEstimate — Day 39 / 100
## Research-grade dataset audit, ground-truth visualization, ROS stereo discovery and measurement readiness

**Version:** 1.0 · **Language:** Indonesian / English technical terminology · **Environment:** Kaggle Python 3.11+ · **Data source:** `/kaggle/input/datasets/wafabila/fishsizeestimate-dataset`

> **Research objective:** study a defensible pipeline for fish *total length* estimation (cm) from stereo recordings. **This notebook does not fabricate image-to-fish assignments or estimated centimetres.** Its default path is safe and can complete without loading 22.8 GiB of `.bag` data into RAM. A user-controlled optional section reads actual ROS messages and visualizes frames; quantitative length evaluation is deliberately gated on verified correspondence, calibrated geometry and specimen ID.

**Data observed in initial audit:** 37 source files, 11 ROS1 bags (~22.8 GiB combined), 8 calibration YAML files, 3 CSVs, 10 specimen/reference images, 1 tracking JSON, 1 model checkpoint. Ground-truth length table has **9 individuals**. Dataset B source was truncated; only two recovered bags passed ZIP CRC checks. ROS message integrity, stereo synchronization and identity mapping remain **unverified**.

**Workflow**
1. Reproducibility and filesystem audit
2. Provenance, manifest and source integrity checks
3. Ground-truth schema, descriptive statistics and leakage-aware unit-of-analysis
4. Actual specimen/reference images, ground-truth bar chart and contact sheet
5. Tracking annotation structure and calibration inspection
6. ROS bag header and available recording metadata
7. **Optional** ROS message/channel inspection and actual frame sampling (one bag only)
8. Research design, evidence-based evaluation readiness and export artifacts

**Hard scientific boundaries:** object localization ≠ identity association ≠ metric physical length. Image pixels alone cannot establish cm without valid scale/depth. Avoid reading `generic_fish_segmenter.pt` via untrusted pickle deserialization. Do not make random frame-level train/test splits.

## 0. Configuration & reproducibility
Default flags intentionally avoid expensive ROS message decoding; set `INSPECT_ROSBAG=True` after seeing bag inventory. The first run audits and visualizes the ground truth and specimen images with no internet requirement.

In [ ]:
from pathlib import Path
import os, sys, json, csv, re, hashlib, platform, time, warnings
from collections import Counter, defaultdict
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from IPython.display import display, Markdown

DATA_ROOT = Path('/kaggle/input/datasets/wafabila/fishsizeestimate-dataset')
WORK_DIR = Path('/kaggle/working/fishsizeestimate_day39')
WORK_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
np.random.seed(SEED)
INSPECT_ROSBAG = False        # Switch on only after first notebook pass
BAG_INDEX = 0                # Which of 11 available recordings
MAX_IMAGE_MESSAGES = 12      # Safety bound for optional extraction
MAX_TOTAL_MESSAGES = 2500    # Safety bound for optional streaming loop
MAX_FRAME_EDGE = 960         # Preview only; does not alter source

assert DATA_ROOT.is_dir(), f'Dataset missing: {DATA_ROOT}'
print('DATA_ROOT:', DATA_ROOT)
print('WORK_DIR:', WORK_DIR)
print('Python:', sys.version.split()[0], '| platform:', platform.platform())
print('pandas:', pd.__version__, '| numpy:', np.__version__)
print('UTC:', datetime.now(timezone.utc).isoformat())
print('INSPECT_ROSBAG:', INSPECT_ROSBAG)

## 1. Data provenance and immutable inventory
The user-supplied upload is a **partial recovery**: one Dataset B recording from the upstream corrupted ZIP is absent. The 11 available bag headers are not a guarantee of valid messages or usable stereo pairs. Files carry a hash prefix added by the packaging script; original paths are recovered from `source_manifest.json`. **No raw data is copied into `/kaggle/working`.**

In [ ]:
files = sorted((p for p in DATA_ROOT.rglob('*') if p.is_file()), key=lambda p:p.name)
allowed = {'.bag','.csv','.json','.yaml','.yml','.jpg','.jpeg','.png','.webp','.pt','.md'}
records = [{'name':p.name, 'path':str(p), 'type':p.suffix.lower(),
            'size_bytes':p.stat().st_size, 'size_mib':p.stat().st_size/2**20} for p in files]
inv = pd.DataFrame(records)
print('FILES:',len(files),'TOTAL GiB:',round(inv.size_bytes.sum()/2**30,3))
print('Types:',Counter(inv.type))
display(inv[['name','type','size_mib']].sort_values('size_mib',ascending=False).head(20).style.format({'size_mib':'{:,.2f}'}))
assert len(files)>=30, 'Unexpectedly sparse mounted dataset; investigate'
assert not any('__MACOSX' in p.parts or p.name.startswith('._') or '~lock' in p.name for p in files), 'Hidden/lock files found'
inv.to_csv(WORK_DIR/'source_inventory.csv',index=False)

In [ ]:
def find_one(suffix):
    hits=[p for p in files if p.name==suffix or p.name.endswith('_'+suffix)]
    if len(hits)!=1:
        raise FileNotFoundError(f'Expected exactly one {suffix}; found {len(hits)}: {hits}')
    return hits[0]

manifest_file=find_one('source_manifest.json')
manifest=json.loads(manifest_file.read_text(encoding='utf-8'))
manifest_entries=manifest.get('files',[])
by_upload={r['upload_name']:r for r in manifest_entries}
print('Manifest state:',manifest.get('status'))
print('Entries:',len(manifest_entries))
for lim in manifest.get('limitations',[]): print('LIMITATION:',lim)
not_found=[r['upload_name'] for r in manifest_entries if not (DATA_ROOT/r['upload_name']).is_file()]
assert not not_found, f'Manifest files missing: {not_found[:5]}'
# Size audit without hashing 22.8GiB again
mismatch=[r['upload_name'] for r in manifest_entries if (DATA_ROOT/r['upload_name']).stat().st_size!=r['size_bytes']]
assert not mismatch, f'Size mismatch: {mismatch[:5]}'
print('PASS: manifest path + byte-size concordance')

def original_path(p):
    return by_upload.get(p.name,{}).get('original_path',p.name)

print('Sample provenance paths:')
for p in files[:5]: print(' ',p.name,'=>',original_path(p))

## 2. Ground-truth *total length* (cm) and descriptive exploration
**Ground truth unit = individual fish**, not frame. Nine values **cannot** support a credible random train/validation/test split for a deep regression model. The metric field `Total_Length_cm` is the only observed physical target in this upload; no inferred values are substituted.

In [ ]:
gt_file=find_one('fish_gt_measures.csv')
gt=pd.read_csv(gt_file)
required={'Dataset','Identifier','Species_Common_Name','Species_Scientific_Name','Total_Length_cm'}
assert required.issubset(gt.columns), f'Missing columns: {required-set(gt.columns)}'
gt['Total_Length_cm']=pd.to_numeric(gt['Total_Length_cm'],errors='coerce')
assert gt['Total_Length_cm'].notna().all() and (gt['Total_Length_cm']>0).all()
assert not gt.duplicated(['Dataset','Identifier']).any(), 'Duplicate individual IDs'
gt['subject_id']=gt['Dataset'].astype(str)+' / '+gt['Identifier'].astype(str)
print('Individuals:',len(gt),'| dataset membership:',gt['Dataset'].value_counts().to_dict())
print('Species scientific:',gt['Species_Scientific_Name'].value_counts().to_dict())
display(gt)
display(gt.groupby('Dataset').agg(individuals=('subject_id','count'),length_min_cm=('Total_Length_cm','min'),length_mean_cm=('Total_Length_cm','mean'),length_max_cm=('Total_Length_cm','max')).round(2))
gt.to_csv(WORK_DIR/'ground_truth_audited.csv',index=False)

In [ ]:
fig, axes=plt.subplots(1,2,figsize=(15,5),constrained_layout=True)
ordered=gt.sort_values('Total_Length_cm')
axes[0].barh(ordered.subject_id,ordered['Total_Length_cm'])
axes[0].set_xlabel('Measured total length (cm)')
axes[0].set_title('Ground-truth total length — 9 fish (NOT model estimates)')
for i,v in enumerate(ordered['Total_Length_cm']): axes[0].text(v+0.15,i,f'{v:g} cm',va='center',fontsize=8)
axes[0].set_xlim(0,max(gt['Total_Length_cm'])*1.2)
axes[1].hist(gt['Total_Length_cm'],bins=min(6,len(gt)),edgecolor='black')
axes[1].set_xlabel('Measured total length (cm)')
axes[1].set_ylabel('Individuals')
axes[1].set_title('Ground truth distribution — descriptive only')
plt.savefig(WORK_DIR/'ground_truth_lengths.png',dpi=160,bbox_inches='tight')
plt.show()

## 3. Specimen photos: actual input images with ground truth labels
These photos are **reference/specimen images**, not necessarily temporally aligned stereo frames. A name-based association (`Fish_0` ↔ `Fish 0`, `red_tag` ↔ `Red Tag`) can be displayed for human review, **but must not be treated as proven correspondence to bag frame labels**. Every card shows a reported total length, not a length predicted from pixels.

In [ ]:
from matplotlib import pyplot as plt
image_paths=[p for p in files if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]

def parse_reference(p):
    orig=original_path(p)
    name=Path(orig).stem
    dataset='Dataset A' if 'Dataset_A_fish_specimens' in orig else ('Dataset B' if 'Dataset_B_fish_specimens' in orig else None)
    label=None
    if dataset=='Dataset A':
        m=re.fullmatch(r'Fish_(\d+)',name,re.I)
        if m:label='Fish '+m.group(1)
    elif dataset=='Dataset B':
        lookup={'red_tag':'Red Tag','black_tag':'Black Tag','unmarked':'Unmarked'}
        label=lookup.get(name.lower())
    match=gt[(gt.Dataset==dataset)&(gt.Identifier==label)] if dataset and label else gt.iloc[0:0]
    value=float(match.iloc[0]['Total_Length_cm']) if len(match)==1 else None
    return dataset,label,value,orig

rows=[]
for p in image_paths:
    try:
        with Image.open(p) as raw:
            im=ImageOps.exif_transpose(raw)
            dims=im.size
            im.verify() if False else None
        status='readable'
    except Exception as exc:
        dims=None;status=f'failed: {exc}'
    dataset,label,length,orig=parse_reference(p)
    rows.append({'file':p.name,'source_path':orig,'dataset':dataset,'subject_id':label,'gt_length_cm':length,'dimensions':str(dims),'image_status':status})
refs=pd.DataFrame(rows)
display(refs[['dataset','subject_id','gt_length_cm','dimensions','image_status','source_path']])
refs.to_csv(WORK_DIR/'specimen_reference_mapping.csv',index=False)
assert (refs.image_status=='readable').all(), 'Specimen image decode failed'

In [ ]:
cols=4
n=len(image_paths)
fig,axs=plt.subplots((n+cols-1)//cols,cols,figsize=(16, 4*((n+cols-1)//cols)),constrained_layout=True)
axs=np.asarray(axs).reshape(-1)
for i,p in enumerate(image_paths):
    dataset,ident,length,orig=parse_reference(p)
    with Image.open(p) as raw:
        im=ImageOps.exif_transpose(raw).convert('RGB')
        im.thumbnail((550,420))
        axs[i].imshow(im)
    axs[i].axis('off')
    tag=f'{dataset or "Unknown"} / {ident or "group photo"}'
    label=f' | GT {length:g} cm' if length is not None else ' | GT: no 1:1 label'
    axs[i].set_title(tag+label,fontsize=9)
for ax in axs[n:]:ax.axis('off')
fig.suptitle('Actual specimen/reference images (ground-truth lengths from CSV only)',fontsize=14)
plt.savefig(WORK_DIR/'reference_specimen_contact_sheet.png',dpi=150,bbox_inches='tight')
plt.show()

## 4. Dataset A/B summary metadata — verify before assigning video identities
Review the source summary tables verbatim. **Do not derive an individual fish ID from timestamps alone** without a documented mapping. The Dataset B recordings can contain multiple fish; per-frame identity association is a separate supervised/tracking problem.

In [ ]:
summaries={}
for key in ['Dataset_Summary_Dataset_A.csv','Dataset_Summary_Dataset_B.csv']:
    p=find_one(key)
    tab=pd.read_csv(p)
    summaries[key]=tab
    print('\n',key,'shape:',tab.shape,'columns:',tab.columns.tolist())
    display(tab.head(12))
    tab.head(1000).to_csv(WORK_DIR/key,index=False)

## 5. Tracking annotations: inspect schema without inventing image-level labels
The JSON is inspected for top-level types and modest previews. No arbitrary nested keys are assumed to contain physical length. If large, the notebook avoids printing every annotation.

In [ ]:
p=find_one('model_tracking_annotations.json')
tracking=json.loads(p.read_text(encoding='utf-8'))
print('JSON root type:',type(tracking).__name__)
if isinstance(tracking,dict):
    for k,v in tracking.items():
        size=len(v) if isinstance(v,(list,dict,str)) else 'scalar'
        print(' -',str(k),'type:',type(v).__name__,'size:',size)
    print('Sample keys:',list(tracking)[:15])
    for k in list(tracking)[:3]:
        print('Preview',k,':',str(tracking[k])[:500])
elif isinstance(tracking,list):
    print('Total entries:',len(tracking))
    for v in tracking[:3]:print(str(v)[:600])
else:
    print('Preview:',str(tracking)[:500])

## 6. Stereo camera calibration: keys and validity gates
Eight YAML files come from different sessions, left/right cameras. Their flattened upload names are mapped to original locations via the manifest. We parse with `yaml.safe_load` (never unsafe YAML loading) and inspect numeric camera matrices only if present. We do **not** guess baseline units or substitute a stereo baseline.

In [ ]:
try:
    import yaml
except ImportError:
    print('PyYAML unavailable; install in Kaggle or leave calibration parsing as not evaluated')
    yaml=None
calib_files=[p for p in files if p.suffix.lower() in {'.yaml','.yml'}]
calib_docs={}
cal_rows=[]
for p in calib_files:
    name=original_path(p)
    if yaml is None:doc={}
    else:
        try: doc=yaml.safe_load(p.read_text(encoding='utf-8-sig'))
        except Exception as exc:
            doc={};print('CALIBRATION PARSE ERROR',name,exc)
    calib_docs[name]=doc
    keys=sorted(doc) if isinstance(doc,dict) else []
    cal_rows.append({'file':p.name,'original_path':name,'keys':', '.join(keys),'parsed_mapping':isinstance(doc,dict)})
cal=pd.DataFrame(cal_rows)
display(cal[['original_path','parsed_mapping','keys']])
cal.to_csv(WORK_DIR/'calibration_inventory.csv',index=False)

In [ ]:
def get_mat(d,key):
    if not isinstance(d,dict):return None
    val=d.get(key)
    if isinstance(val,dict):val=val.get('data',val.get('values'))
    if not isinstance(val,(list,tuple)):return None
    a=np.asarray(val,dtype=float)
    if a.size==9:return a.reshape(3,3)
    if a.size==12:return a.reshape(3,4)
    return None

param_rows=[]
for name,doc in calib_docs.items():
    camera=get_mat(doc,'camera_matrix')
    projection=get_mat(doc,'projection_matrix')
    # Flexible but non-assumptive: do not infer translation unit
    size=(doc.get('image_width'),doc.get('image_height')) if isinstance(doc,dict) else (None,None)
    param_rows.append({'original_path':name,'image_width':size[0],'image_height':size[1],
                       'fx_px':float(camera[0,0]) if camera is not None else np.nan,
                       'fy_px':float(camera[1,1]) if camera is not None else np.nan,
                       'cx_px':float(camera[0,2]) if camera is not None else np.nan,
                       'projection_present':projection is not None,
                       'camera_matrix_present':camera is not None})
params=pd.DataFrame(param_rows)
display(params)
params.to_csv(WORK_DIR/'calibration_parameters_inspected.csv',index=False)
print('Caution: these files do not automatically establish rectification, baseline units or a valid stereo pair.')

## 7. ROS `.bag` inventory: actual recording source and safety checks
Default: **read file headers only** (tiny I/O). For valid message streams and topics, enable optional section below. `#ROSBAG V2.0` confirms the magic signature but **does not** verify all chunks/messages. Dataset B has known unrecoverable source data.

In [ ]:
bags=sorted((p for p in files if p.suffix.lower()=='.bag'),key=lambda p:p.name)
bag_rows=[]
for p in bags:
    with p.open('rb') as f: magic=f.read(13)
    orig=original_path(p)
    setname='Dataset B' if 'Dataset_B/' in orig else ('Dataset A' if 'Dataset_A/' in orig else 'Unknown')
    bag_rows.append({'file':p.name,'original_path':orig,'dataset':setname,'size_gib':p.stat().st_size/2**30,'ros_v2_magic':magic.startswith(b'#ROSBAG V2.0')})
bagdf=pd.DataFrame(bag_rows)
display(bagdf[['dataset','original_path','size_gib','ros_v2_magic']].style.format({'size_gib':'{:,.3f}'}))
print('Bags:',len(bags),'Valid ROS V2 headers:',bagdf.ros_v2_magic.sum())
assert len(bags)==11 and bool(bagdf.ros_v2_magic.all()), 'ROS bag inventory does not match expected 11 files'
bagdf.to_csv(WORK_DIR/'bag_inventory.csv',index=False)

## 8. OPTIONAL: Inspect topics and decode a handful of real frames

**Enable explicitly** by setting `INSPECT_ROSBAG = True` in the first code cell, then rerunning this section. Opening a 1–3 GiB ROS bag can take time; **no extraction to disk** by default. Uses the pure-Python [`rosbags` AnyReader](https://ternaris.gitlab.io/rosbags/topics/highlevel.html) and deserializes only selected image topics. Depending on the bag and Kaggle wheel availability, this step can fail without invalidating the primary audit. Never call `torch.load` on unknown `.pt` files. 

The extraction is capped by message and preview limits and emits no fabricated frames. Stereo channels are discovered from **actual recorded topic names**. A pair of preview frames does not prove synchronized stereo or support physical measurements.

In [ ]:
if INSPECT_ROSBAG:
    try:
        from rosbags.highlevel import AnyReader
    except ImportError:
        print('rosbags missing. Install using: %pip install rosbags')
        raise
    selected=bagdf.iloc[BAG_INDEX]
    chosen=next(p for p in bags if p.name==selected['file'])
    print('Selected bag:',selected['original_path'])
    with AnyReader([chosen]) as reader:
        topic_table=[{'topic':c.topic,'message_type':c.msgtype,'messages':c.msgcount} for c in reader.connections]
        topic_df=pd.DataFrame(topic_table)
        display(topic_df.sort_values('messages',ascending=False) if len(topic_df) else topic_df)
        topic_df.to_csv(WORK_DIR/'ros_topics_selected_bag.csv',index=False)
        image_conns=[c for c in reader.connections if ('CompressedImage' in c.msgtype or c.msgtype.endswith('/Image'))]
        print('Image connections:',[(c.topic,c.msgtype,c.msgcount) for c in image_conns])
        print('Reader time bounds (ns):',reader.start_time,reader.end_time)
else:
    print('SKIP: INSPECT_ROSBAG=False; no multi-gigabyte ROS file read. Set True after primary audit.')

In [ ]:
def decode_ros_image(msg, msgtype):
    """Decode standard ROS1 compressed/raw 8-bit imagery only; no arbitrary formats."""
    import cv2
    if 'CompressedImage' in msgtype:
        arr=np.frombuffer(bytes(msg.data),dtype=np.uint8)
        bgr=cv2.imdecode(arr,cv2.IMREAD_COLOR)
        if bgr is None:return None
        return cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB)
    if msgtype.endswith('/Image'):
        encoding=str(getattr(msg,'encoding','')).lower()
        raw=np.frombuffer(bytes(msg.data),dtype=np.uint8)
        height,width,step=int(msg.height),int(msg.width),int(msg.step)
        if width<=0 or height<=0 or step<=0 or raw.size<height*step:return None
        plane=raw[:height*step].reshape(height,step)
        if encoding in ('rgb8','bgr8') and step>=width*3:
            frame=plane[:,:width*3].reshape(height,width,3)
            return frame if encoding=='rgb8' else frame[:,:,::-1]
        if encoding=='mono8' and step>=width:
            return np.repeat(plane[:,:width,None],3,axis=2)
    return None

frame_samples=[]
if INSPECT_ROSBAG:
    import cv2
    selected=bagdf.iloc[BAG_INDEX]
    chosen=next(p for p in bags if p.name==selected['file'])
    with AnyReader([chosen]) as reader:
        image_conns=[c for c in reader.connections if ('CompressedImage' in c.msgtype or c.msgtype.endswith('/Image'))]
        if not image_conns:
            print('No supported ROS image topics; inspect topic table above.')
        else:
            # Prefer at most two likely left/right image streams; retain actual names
            chosen_conns=image_conns[:2]
            msg_count=0
            for conn,ts,data in reader.messages(connections=chosen_conns):
                msg_count+=1
                if msg_count>MAX_TOTAL_MESSAGES or len(frame_samples)>=MAX_IMAGE_MESSAGES:break
                try:
                    msg=reader.deserialize(data,conn.msgtype)
                    frame=decode_ros_image(msg,conn.msgtype)
                    if frame is None:continue
                    h,w=frame.shape[:2]
                    if max(h,w)>MAX_FRAME_EDGE:
                        ratio=MAX_FRAME_EDGE/max(h,w)
                        frame=cv2.resize(frame,(max(1,int(w*ratio)),max(1,int(h*ratio))))
                    frame_samples.append({'topic':conn.topic,'timestamp_ns':int(ts),'rgb':frame,'source':chosen.name})
                except Exception as exc:
                    print('Frame decode warning:',conn.topic,type(exc).__name__,str(exc)[:120])
            print('Scanned messages:',msg_count,'Decoded frames:',len(frame_samples))
            if frame_samples:
                n=min(len(frame_samples),6)
                fig,axs=plt.subplots(2,3,figsize=(15,9),constrained_layout=True)
                for ax in axs.ravel():ax.axis('off')
                for ax,item in zip(axs.ravel(),frame_samples[:n]):
                    ax.imshow(item['rgb']);ax.set_title(item['topic']+'\n'+str(item['timestamp_ns']),fontsize=9)
                plt.savefig(WORK_DIR/'sample_real_ros_frames.png',dpi=140,bbox_inches='tight')
                plt.show()
else:
    print('SKIP: no ROS image messages read.')

## 9. Measurement methodology: what must be established before centimetre regression

### Scientific model of the task
- **Observed labels:** total length (cm) for nine individuals (not necessarily frame-level visible endpoints).
- **Potential observations:** left/right images from ROS topics, with times, session and calibration.
- **Required correspondence:** image/frame → recording/session → fish ID → one measured length. Dataset B may contain multiple fish: this cannot be assumed.
- **Metric scale:** calibrated stereo geometry and reliable pixel correspondence, or a verified in-frame scale object. Bounding-box pixel width alone is **not** total length in cm.
- **Geometry:** for rectified stereo with measured disparity `d` in pixels, validated focal length `fx` in pixels, and camera baseline `B` in meters, approximate depth is `Z = fx * B / d` (only for valid pixels, nonzero disparity and correct rectification). For physical fish total length, estimate 3D snout and tail-tip points, then compute Euclidean separation; body curvature and occlusion need special handling. Units, distortion, calibration date and timestamp alignment must be verified first.
- **Model options:** (a) geometric stereo keypoints/segmentation; (b) pretrained segmenter + postprocessing + calibrated 3D extent; (c) supervised regressor **only after** per-individual or per-session pairing is verified.

### Anti-leakage protocol
1. Preserve every frame's original bag and timestamp.
2. Build an explicit auditable identity mapping; leave uncertain frames unlabeled.
3. Group by **fish ID** and acquisition session; never split adjacent frames randomly.
4. Use training/validation for model design; reserve held-out test identities/sessions if enough individuals exist. With only nine measured individuals, do not overstate metric generalization.
5. Report MAE (cm), RMSE (cm), median AE (cm), error by species/session, sample counts, and confidence intervals **only for verified paired predictions**.
6. Record segmentation failure rate and stereo pairing/invalid-depth rate rather than silently discarding failed cases.

**Current result status:** `NOT YET EVALUATED`; no numeric metric is computed in this notebook without verified frame-level predicted length pairs.

## 10. Evaluation contract & readiness gates (strict)
Each flag is tied to an observable source or a **manual user audit**. This notebook only asserts machine-verifiable upstream counts; it will not mark fish-to-bag mapping, calibration baseline, stereo synchronization, endpoint annotation, or final test performance as verified by inference.

In [ ]:
gates={
    'Kaggle mounted dataset':DATA_ROOT.is_dir(),
    'Manifest files and sizes matched':not (not_found or mismatch),
    'Ground truth present and numeric':len(gt)>0 and gt['Total_Length_cm'].notna().all(),
    '11 ROS1 bag headers present':len(bags)==11 and bool(bagdf.ros_v2_magic.all()),
    '8 calibration YAML files present':len(calib_files)==8,
    'Specimen images readable':bool((refs.image_status=='readable').all()),
    'Full ROS message integrity verified':False,
    'Left/right topic synchronization verified':False,
    'Calibration session matched and baseline units checked':False,
    'Frame to fish-identity mapping confirmed':False,
    'Fish nose/tail endpoints or metric scale annotations verified':False,
    'Defensible identity/session-disjoint test performed':False,
    'Held-out physical length MAE/RMSE':False
}
gate_df=pd.DataFrame([{'criterion':k,'verified':bool(v)} for k,v in gates.items()])
display(gate_df)
gate_df.to_csv(WORK_DIR/'readiness_gates.csv',index=False)
print(f'Automatically verified: {sum(gates.values())}/{len(gates)} — remaining scientific gates explicitly open')

## 11. Baselines and fair experimentation (not fabricated)

**Baseline A — constant length predictor:** a meaningful *regression* baseline can be evaluated only once explicit independent examples and a defensible group split exist. Predicting the mean of all nine fish on those same fish would leak labels, so we do not do that here.

**Baseline B — pixel span only:** optional 2D geometric size **in pixels**, not centimetres; useful for checking segmentation or endpoint stability, but not as a valid metric fish length estimator.

**Champion selection:** no champion is declared. A checkpoint `generic_fish_segmenter.pt` exists, but its architecture, provenance and inference contract have not been audited; do not load via unsafe pickling in this default audit. Once a trusted model is established, use it for candidate masks, verify geometric calibration and compare length estimates to held-out ground truth.

**Do not claim accuracy** based on reading a bag, plotting specimen photos or matching a model filename.

## 12. Report generation and Kaggle outputs
Write traceable artifacts under `/kaggle/working/fishsizeestimate_day39/`, which can be collected from **Save Version → Output**. No source data is modified. Avoid exporting proprietary/unlicensed photos into public distribution without reviewing upstream rights.

In [ ]:
report={
    'project':'FishSizeEstimate Day 39',
    'audit_generated_utc':datetime.now(timezone.utc).isoformat(),
    'dataset_root':str(DATA_ROOT),
    'file_count':len(files),
    'size_gib':round(sum(p.stat().st_size for p in files)/2**30,4),
    'bag_count':len(bags),
    'ros_v2_headers_valid':int(bagdf.ros_v2_magic.sum()),
    'gt_individuals':int(len(gt)),
    'gt_numeric_target':'Total_Length_cm',
    'calibration_yaml_count':len(calib_files),
    'specimen_image_count':len(image_paths),
    'upstream_manifest_status':manifest.get('status'),
    'known_upstream_limitations':manifest.get('limitations',[]),
    'optional_ros_inspection_enabled':bool(INSPECT_ROSBAG),
    'quantitative_length_evaluation':'NOT YET EVALUATED',
    'evaluation_reason':'No verified per-frame fish-identity correspondence and metric stereo length predictions',
    'readiness_gates':gates
}
(WORK_DIR/'audit_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
readme="""# FishSizeEstimate Day 39 — generated audit artifacts

Generated by the Kaggle notebook. This is a research audit, not a trained metric-length model.

- `source_inventory.csv`: source files and sizes
- `ground_truth_audited.csv`: nine measured individuals (cm)
- `ground_truth_lengths.png`: descriptive GT visualization
- `reference_specimen_contact_sheet.png`: source specimen/reference images
- `specimen_reference_mapping.csv`: name-based reference mapping for manual review
- `calibration_inventory.csv`, `calibration_parameters_inspected.csv`: camera calibration inspection
- `bag_inventory.csv`: actual ROS source recordings and header checks
- `readiness_gates.csv`, `audit_report.json`: evidence and open gates
- Optional: `ros_topics_selected_bag.csv`, `sample_real_ros_frames.png`

**Known limitation:** Dataset B was partially recovered; one upstream recording absent. Nine ground-truth fish do not prove a frame-to-identity match. Physical length evaluation remains NOT YET EVALUATED. Raw data is not recopied.
"""
(WORK_DIR/'README.md').write_text(readme,encoding='utf-8')
outputs=sorted(p for p in WORK_DIR.iterdir() if p.is_file())
print('OUTPUT DIRECTORY:',WORK_DIR)
for p in outputs: print(f' - {p.name} ({p.stat().st_size/1024:.1f} KiB)')
print('Saved reports are actual results of audit only. No fabricated model scores.')

## 13. Roadmap, limitations, and references

### Before → Now → Next
**Before:** Zenodo metadata plus 23.45 GiB nested files; source `multiple_fish.zip` truncated at 4,294,967,295 bytes. Nine bag files from Dataset A and two CRC-matched bag files from Dataset B were recovered. Kaggle upload ultimately succeeded.

**Now:** 37 mounted files; 11 ROS V2 headers; 9 ground-truth individuals, reference images, YAML calibration, tracking JSON and reproducible audit exports. **No per-frame metric length predictions.**

**Next (in order):**
1. Inspect topics and messages of one bag with `INSPECT_ROSBAG=True`; validate actual ROS message count and decompressibility.
2. Check camera calibration schema, date, baseline and left/right topic mapping; visualize synchronized stereo frames with timestamps.
3. Build an identity mapping table with original paths, frame timestamps, specimen identifier, and measured length; leave uncertain observations out.
4. Establish geometric baseline (manual endpoint keypoints and valid stereo calibration) and failure-rate logging.
5. Only then train candidate estimators or evaluate provided segmenter in a sandboxed trusted runtime.
6. Group split by identity/session; select on validation; calculate final held-out metrics exactly once, include per-session/species breakdowns and limitations.
7. Export reproducible model + documented FastAPI/frontend implementation after research evidence exists.

### Limitations
Small count of labeled individuals (n=9); Dataset B partial recovery; lack of independently verified per-frame fish identity labels; unknown stereo calibration/extrinsic units until inspected; class/model licensing and `.pt` trust status require verification; ROS bag header alone is not full file integrity.

### Technical references
- [`rosbags` high-level AnyReader documentation](https://ternaris.gitlab.io/rosbags/topics/highlevel.html): ROS1/ROS2 bag reading and deserialization.
- [OpenCV stereo depth tutorial](https://docs.opencv.org/3.4.20/dd/d53/tutorial_py_depthmap.html): disparity-based stereo geometry.
- User-supplied `dataset_to_zenodo/Dataset_description.md` within uploaded source (available as flattened file with path in manifest).

### Provenance boundary
Source-specific DOI, license grant and public distribution permission **have not yet been independently verified**. See uploaded original description and rights statement before publicly redistributing datasets or model checkpoint.